In [1]:
import cProfile
import pstats
import io

def profile_function(func):
    """Decorator to profile a function."""
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        sortby = 'cumulative'
        ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
        ps.print_stats()
        print(s.getvalue())
        return result
    return wrapper


In [2]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime, timedelta
import time
import csv
import asyncio
import nest_asyncio
import logging
import websocket as ws_client
import xarray as xr
import threading

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)

usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [3]:
import pandas as pd
fno_scrips_mcx = pd.read_csv('//home/deep/Desktop/NEWshoonya/MCX_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
fno_scrips_mcx['StrikePrice'] = fno_scrips_mcx['StrikePrice'].astype(float)
fno_scrips_mcx.sort_values('Expiry',inplace=True)
fno_scrips_mcx.reset_index(drop=True, inplace=True)

fno_scrips = pd.read_csv('/home/deep/Desktop/NEWshoonya/NFO_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])
fno_scrips['StrikePrice'] = fno_scrips['StrikePrice'].astype(float)
fno_scrips.sort_values('Expiry',inplace=True)
fno_scrips.reset_index(drop=True, inplace=True)

/tmp/ipykernel_9469/181564701.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
/tmp/ipykernel_9469/181564701.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])


In [4]:
import numpy as np
from numba import njit

@njit
def ema_numba(close, length):
    out = np.full_like(close, np.nan)
    
    if len(close) <= length:
        return out
    
    alpha = 2 / (length + 1)
    ema = np.mean(close[:length])  # Initialize with SMA
    out[length-1] = ema
    
    for i in range(length, len(close)):
        ema = alpha * close[i] + (1 - alpha) * ema
        out[i] = ema
    
    return np.round(out, 2)

import numpy as np
from numba import njit

@njit
def jma_numba(src, length, phase, power):
    out = np.full_like(src, np.nan)
    
    if len(src) <= length:
        return out
    
    phaseRatio = 1.5 + phase / 100 if -100 <= phase <= 100 else (0.5 if phase < -100 else 2.5)
    beta = 0.45 * (length - 1) / (0.45 * (length - 1) + 2)
    alpha = np.power(beta, power)
    
    e0 = e1 = e2 = jma = 0.0
    
    for i in range(len(src)):
        if i == 0:
            e0 = src[i]
            e1 = 0.0
            e2 = 0.0
            jma = 0.0
        else:
            e0 = (1 - alpha) * src[i] + alpha * e0
            e1 = (src[i] - e0) * (1 - beta) + beta * e1
            e2 = (e0 + phaseRatio * e1 - jma) * np.power(1 - alpha, 2) + np.power(alpha, 2) * e2
            jma = e2 + jma
            
            # Reset values if they become too large or unstable
            if np.abs(jma) > 1e10:
                e0 = e1 = e2 = jma = 0.0
        
        if i >= length - 1:
            out[i] = np.round(jma, 2)
    
    return out

import numpy as np
from numba import njit

@njit
def alma_numba(series, length, offset, sigma):
    out = np.full_like(series, np.nan)
    
    if len(series) < length:
        return out
    
    for i in range(length - 1, len(series)):
        numerator = 0.0
        denominator = 0.0
        m = offset * (length - 1)
        s = length / sigma
        for j in range(length):
            weight = np.exp(-((j - m) ** 2) / (2 * s * s))
            numerator += weight * series[i - length + 1 + j]
            denominator += weight
        out[i] = round(numerator / denominator, 2)
    
    return out


@njit
def atr_numba(high, low, close, length):
    true_range = np.zeros_like(close)
    atr = np.full_like(close, np.nan)
    
    for i in range(1, len(close)):
        true_range[i] = max(high[i] - low[i], abs(high[i] - close[i-1]), abs(low[i] - close[i-1]))
    
    for i in range(length, len(close)):
        if i == length:
            atr[i] = np.mean(true_range[:length])
        else:
            atr[i] = (atr[i - 1] * (length - 1) + true_range[i]) / length
    
    return atr


@njit
def rsi_trail_numba(open, high, low, close, rsi_lower=45, rsi_upper=55, ma_length=20, ma_offset=0.85, ma_sigma=6):
    ohlc4 = (open + high + low + close) / 4
    ma = alma_numba(ohlc4, ma_length, ma_offset, ma_sigma)
    atr = atr_numba(high, low, close, 7)
    upper_bound = ma + (rsi_upper - 50) / 10 * atr
    lower_bound = ma - (50 - rsi_lower) / 10 * atr
    
    signal = np.zeros_like(close, dtype=np.float64)
    is_bullish = np.zeros_like(close, dtype=np.bool_)
    is_bearish = np.zeros_like(close, dtype=np.bool_)
    
    signal[:ma_length] = np.nan  # Set initial values to NaN
        
    for i in range(ma_length, len(close)):
        if ohlc4[i] > upper_bound[i]:
            if not is_bullish[i-1]:
                signal[i] = 1  # Bullish crossover
            is_bullish[i] = True
            is_bearish[i] = False
        elif close[i] < lower_bound[i]:
            if not is_bearish[i-1]:
                signal[i] = -1  # Bearish crossunder
            is_bullish[i] = False
            is_bearish[i] = True
        else:
            signal[i] = 0  # Neutral
            is_bullish[i] = is_bullish[i-1]
            is_bearish[i] = is_bearish[i-1]
    
    return signal, is_bullish, is_bearish

@njit
def chandelier_exit_numba(open, high, low, close, length=2, mult=1):
    atr = atr_numba(high, low, close, length) * mult
    
    long_stop = np.full_like(close, np.nan)
    short_stop = np.full_like(close, np.nan)
    direction = np.zeros_like(close)
    
    for i in range(length, len(close)):
        highest = np.max(close[i-length+1:i+1])
        lowest = np.min(close[i-length+1:i+1])
        
        long_stop[i] = highest - atr[i]
        short_stop[i] = lowest + atr[i]
        
        if i > length:
            long_stop[i] = max(long_stop[i], long_stop[i-1]) if close[i-1] > long_stop[i-1] else long_stop[i]
            short_stop[i] = min(short_stop[i], short_stop[i-1]) if close[i-1] < short_stop[i-1] else short_stop[i]
        
        if close[i] > short_stop[i-1]:
            direction[i] = 1
        elif close[i] < long_stop[i-1]:
            direction[i] = -1
        else:
            direction[i] = direction[i-1] if i > 0 else 0
    
    return long_stop, short_stop, direction

@njit
def calculate_last_5_bars_length(high, low):
    # Calculate the length of each bar
    bar_lengths = high - low
    # Calculate the average length of the last 10 bars
    last_5_bars_length = np.mean(bar_lengths[-5:])
    return last_5_bars_length

@njit
def calculate_current_bar_length(high, low):
    # Calculate the length of the current bar
    current_bar_length = high[-1] - low[-1]
    return current_bar_length


In [5]:
symbol = 'BANKNIFTY'
class TradingState:
    def __init__(self):
        self.ce_trading_symbol = None
        self.pe_trading_symbol = None
        self.ce_trading_token = None
        self.pe_trading_token = None

state = TradingState()


async def find_atm_strike_and_symbols():   
    global  atm_strike
    cmp_bn = float(resampled_data[ini]['close'].iloc[-1])      
    
    # In a real scenario, you might want to fetch this price asynchronously
    #cmp_bn = await get_current_price()  # You'd need to implement this function
    
    mod = int(cmp_bn) % 100
    cmp_bn = int(cmp_bn)
    atm_strike = cmp_bn - mod if mod <= 50 else cmp_bn + (100 - mod)
    #print(atm_strike)

    
    # Assuming fno_scrips_mcx is a global DataFrame
    filtered_df = fno_scrips[
        (fno_scrips['Symbol'] == symbol) &
        (fno_scrips['StrikePrice'] == float(atm_strike))
    ]
    
    if filtered_df.empty:
        print(f"Could not find trading symbols for ATM strike {atm_strike}")
        return None, None, None, None

    ce_row = filtered_df[filtered_df['OptionType'] == 'CE'].sort_values('Expiry').iloc[0]
    pe_row = filtered_df[filtered_df['OptionType'] == 'PE'].sort_values('Expiry').iloc[0]

    ce_trading_symbol, pe_trading_symbol = ce_row['TradingSymbol'], pe_row['TradingSymbol']
    ce_trading_token, pe_trading_token = ce_row['Token'], pe_row['Token']

    return ce_trading_symbol, pe_trading_symbol, ce_trading_token, pe_trading_token

    

async def update_atm_symbols():
    while True:
        try:
            result = await find_atm_strike_and_symbols()
            if result[0] is not None:
                (state.ce_trading_symbol, state.pe_trading_symbol, 
                 state.ce_trading_token, state.pe_trading_token) = result
                #print(f"Global symbols updated - CE: {state.ce_trading_symbol}, PE: {state.pe_trading_symbol}")
            else:
                print("Failed to update symbols")
        except Exception as e:
            print(f"Error in update_atm_symbols: {e}")
        await asyncio.sleep(5)

In [6]:
async def trading_logic():
    global last_processed_candle, resampled_data
    while trading_active:
        current_time = pd.Timestamp.now()
        
        for token, data in resampled_data.items():
            if not data.empty:
                last_candle_start = data.index[-1]
                
                # Use a fixed 60-second interval if resample_freq is None
                resample_freq = data.index.freq or pd.Timedelta(seconds=60)
                
                current_candle_end = last_candle_start + resample_freq
                
                # Process the data only after the current candle has closed
                if current_time >= current_candle_end:
                    if token not in last_processed_candle or current_time >= last_processed_candle[token] + resample_freq:
                        last_processed_candle[token] = current_candle_end
                        process_token_data(token)

        await asyncio.sleep(0.1)  # Check every 10ms
    

# Function to log trades
def log_trade(timestamp, symbol, action, price, action_time):
    with open("trade_log.csv", "a") as log_file:
        log_file.write(f"{timestamp},{symbol},{action},{price},{action_time}\n")


def get_latest_price(entry_instrument):
    entry_instrument_str = str(entry_instrument)
    with resampling_lock:
        if entry_instrument_str in extra_feedJson:
            latest_data = extra_feedJson[entry_instrument_str][-1]
            latest_price = latest_data['ltp']
            
            return latest_price
        else:
            print(f"{entry_instrument_str} not found in extra_feedJson")  # Debug print
            return None
    

def calculate_retracement_price(high, low, direction):
    if direction == 1:
        # Calculate 50% retracement from high to low
        return high - (high - low) * 0.5
    else:
        # Calculate 50% retracement from low to high
        return low + (high - low) * 0.5


def process_token_data(token):
    df = resampled_data[token]    

    if len(df) > 5:  # Assuming minimum length for Chandelier Exit calculation

        # Convert the necessary columns to numpy arrays
        high = df['high'].values
        low = df['low'].values
        
        # Calculate the average bar length of the last 10 bars
        last_5_bars_length = calculate_last_5_bars_length(high, low)
        print("last 5 bar: ", last_5_bars_length)

        # Calculate the current bar's length
        current_bar_length = calculate_current_bar_length(high, low)
        print("current length: ", current_bar_length)
        print("current time: ", pd.Timestamp.now())

        # Extract the latest and previous direction values
        current_direction = df['ce_direction'].iloc[-1]
        previous_direction = df['ce_direction'].iloc[-2]

        timestamp = df.index[-1]
        
        current_position = current_positions.get(token)
        action_taken = False

        # First, check for exit signals
        if current_position == 'long' and current_direction == -1 and previous_direction == 1:
            # Long exit
            op_token = entry_instruments.get(token, {}).get('op_token')
            op_symbol = entry_instruments.get(token, {}).get('op_symbol')
            
            price = get_latest_price(op_token)
            if price is not None:
                action_time = pd.Timestamp.now()
                log_trade(timestamp, op_symbol, "LONG_EXIT", price, action_time)
                print(f"Long Exit Signal for {op_symbol} at {action_time}")
                current_positions[token] = None
                action_taken = True

        elif current_position == 'short' and current_direction == 1 and previous_direction == -1:
            # Short exit
            op_token = entry_instruments.get(token, {}).get('op_token')
            op_symbol = entry_instruments.get(token, {}).get('op_symbol')
            
            price = get_latest_price(op_token)
            if price is not None:
                action_time = pd.Timestamp.now()
                log_trade(timestamp, op_symbol, "SHORT_EXIT", price, action_time)
                print(f"Short Exit Signal for {op_symbol} at {action_time}")
                current_positions[token] = None
                action_taken = True

        # Then, check for entry signals if we've just exited a position or if we're not in a position
        if action_taken or current_positions.get(token) is None:
            
            if current_direction == 1 and previous_direction == -1 and current_bar_length < (last_5_bars_length * bar_length_multiplier):
                signal_bar_high = high[-1]
                signal_bar_low = low[-1]

                entry_instruments[token] = {
                    'op_token': state.ce_trading_token,
                    'op_symbol': state.ce_trading_symbol
                }

                price = get_latest_price(state.ce_trading_token)
                
                if price is not None:
                    action_time = pd.Timestamp.now()
                    log_trade(timestamp, state.ce_trading_symbol, "LONG ENTRY", price, action_time)
                    print(f"Long Entry Signal for {state.ce_trading_symbol} at {action_time}")
                    current_positions[token] = 'long'
                    entry_prices[state.ce_trading_symbol] = price

            elif current_direction == -1 and previous_direction == 1 and current_bar_length < (last_5_bars_length * bar_length_multiplier):
                signal_bar_high = high[-1]
                signal_bar_low = low[-1]
                
                entry_instruments[token] = {
                    'op_token': state.pe_trading_token,
                    'op_symbol': state.pe_trading_symbol
                }

                price = get_latest_price(state.pe_trading_token)

                if price is not None:
                    action_time = pd.Timestamp.now()
                    log_trade(timestamp, state.pe_trading_symbol, "SHORT ENTRY", price, action_time)
                    print(f"Short Entry Signal for {state.pe_trading_symbol} at {action_time}")
                    current_positions[token] = 'short'
                    entry_prices[state.pe_trading_symbol] = price

            elif (current_direction == 1 and previous_direction == -1 and current_bar_length > (last_5_bars_length * bar_length_multiplier)) or \
                 (current_direction == -1 and previous_direction == 1 and current_bar_length > (last_5_bars_length * bar_length_multiplier)):    
                
                retracement_prices = {}
                signal_bar_high = high[-1]
                signal_bar_low = low[-1]
                retracement_prices[token] = calculate_retracement_price(signal_bar_high, signal_bar_low, current_direction)

                def monitor_retracement():
                    initial_direction = current_direction  # Capture the direction at the time of signal
                    retracement_price = retracement_prices[token]

                    while True:
                        # Fetch the latest price
                        current_price = resampled_data[token]['close'].iloc[-1]

                        # Check if price has retraced to the calculated retracement price
                        retracement_condition = (
                            initial_direction == 1 and current_price <= retracement_price or
                            initial_direction == -1 and current_price >= retracement_price
                        )

                        # Check for direction change
                        if current_direction != initial_direction:
                            print("Direction changed, stopping monitoring.")
                            break

                        if retracement_condition:
                            print("Retracement condition met, entering trade.")
                            ################################################
                            if initial_direction == 1:
                                signal_bar_high = high[-1]
                                signal_bar_low = low[-1]

                                entry_instruments[token] = {
                                    'op_token': state.ce_trading_token,
                                    'op_symbol': state.ce_trading_symbol
                                }

                                price = get_latest_price(state.ce_trading_token)
                                
                                if price is not None:
                                    action_time = pd.Timestamp.now()
                                    log_trade(timestamp, state.ce_trading_symbol, "RETRACE LONG ENTRY", price, action_time)
                                    print(f"Long Entry Signal for {state.ce_trading_symbol} at {action_time}")
                                    current_positions[token] = 'long'
                                    entry_prices[state.ce_trading_symbol] = price

                            elif initial_direction == -1:
                                signal_bar_high = high[-1]
                                signal_bar_low = low[-1]
                                
                                entry_instruments[token] = {
                                    'op_token': state.pe_trading_token,
                                    'op_symbol': state.pe_trading_symbol
                                }

                                price = get_latest_price(state.pe_trading_token)

                                if price is not None:
                                    action_time = pd.Timestamp.now()
                                    log_trade(timestamp, state.pe_trading_symbol, "RETRACE SHORT ENTRY", price, action_time)
                                    print(f"Short Entry Signal for {state.pe_trading_symbol} at {action_time}")
                                    current_positions[token] = 'short'
                                    entry_prices[state.pe_trading_symbol] = price
                            ####################################################
                            break

                        time.sleep(0.1)  # Adjust sleep time as needed

                # Start the monitoring function in a separate thread
                monitoring_thread = threading.Thread(target=monitor_retracement)
                monitoring_thread.start()


In [7]:
def realtime_exit_checker(
    current_positions,
    entry_instruments,
    profit_target,
    stop_loss,
    extra_feedJson
):
    def check_exits():
        while True:
            for token, position in current_positions.items():
                if position is None:
                    continue

                instrument_info = entry_instruments.get(token, {})
                op_token = instrument_info.get('op_token')
                op_symbol = instrument_info.get('op_symbol')

                if op_token is None or op_symbol is None:
                    continue

                current_price = get_latest_price(op_token)
                if current_price is None:
                    continue

                # Get the entry price from our new dictionary
                entry_price = entry_prices.get(op_symbol)
                if entry_price is None:
                    continue

                # Initialize trailing stop loss specific for this position
                trailing_stop_loss = stop_loss

                if position == 'long':
                    profit = current_price - entry_price

                    # Trailing stop logic
                    if profit >= 2:  # Price has moved 20 points in favor
                        # Update trailing stop loss to entry price or higher
                        new_stop_loss = entry_price
                        if new_stop_loss > trailing_stop_loss:
                            trailing_stop_loss = new_stop_loss

                    # Further adjust trailing stop loss as the price increases
                    if current_price - entry_price > 2:
                        new_trailing_stop = current_price - 2
                        if new_trailing_stop > trailing_stop_loss:
                            trailing_stop_loss = new_trailing_stop

                    # Exit if profit target or stop loss is hit
                    if profit >= profit_target or current_price <= trailing_stop_loss:
                        execute_exit(token, op_token, op_symbol, current_price, "REALTIME_LONG_EXIT")

                elif position == 'short':
                    profit = current_price - entry_price

                    # Trailing stop logic
                    if profit >= 2:  # Price has moved 20 points in favor
                        # Update trailing stop loss to entry price or lower 
                        new_stop_loss = entry_price
                        if new_stop_loss > trailing_stop_loss:
                            trailing_stop_loss = new_stop_loss

                    # Further adjust trailing stop loss as the price decreases
                    if current_price - entry_price > 2:
                        new_trailing_stop = current_price - 2
                        if new_trailing_stop > trailing_stop_loss:
                            trailing_stop_loss = new_trailing_stop

                    # Exit if profit target or stop loss is hit
                    if profit >= profit_target or current_price <= trailing_stop_loss:
                        execute_exit(token, op_token, op_symbol, current_price, "REALTIME_SHORT_EXIT")

            time.sleep(0.01)  # Check every 10ms, adjust as needed

    def execute_exit(token, op_token, op_symbol, price, exit_type):
        df = resampled_data[token]
        action_time = pd.Timestamp.now()
        timestamp = df.index[-1]
        log_trade(timestamp, op_symbol, exit_type, price, action_time)
        print(f"{exit_type} for {op_symbol} at {action_time} price: {price}")
        current_positions[token] = None
        entry_instruments.pop(token, None)
        entry_prices.pop(op_symbol, None)  # Remove the entry price when exiting

    exit_thread = threading.Thread(target=check_exits, daemon=True)
    exit_thread.start()


In [ ]:
import asyncio
import threading
from datetime import datetime
import pandas as pd
import nest_asyncio
import time
from concurrent.futures import ThreadPoolExecutor

feed_lock = threading.Lock()

ema_length = 22
last_processed_candle = {}
trading_active = True
feed_opened = False
feedJson = {}
extra_feedJson = {}
resample_frequency = "60s"
resampling_lock = threading.Lock()
last_resample_time = {}
resampled_data = {}
current_positions = {}
# Dictionary to keep track of entry instruments
entry_instruments = {}
entry_prices = {}

profit_target = 100
stop_loss = 20
bar_length_multiplier = 1.5

global signal_bar_high , signal_bar_low 

from concurrent.futures import ThreadPoolExecutor


ini = '26009'  ####################################### intial tokens number

if 'resampled_data' not in globals():
    resampled_data = {}
if 'last_resample_time' not in globals():
    last_resample_time = {}


def event_handler_feed_update(tick_data):
    
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with resampling_lock:
                if token == ini:
                    if token not in feedJson:
                        feedJson[token] = []
                        last_resample_time[token] = timest

                    feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
                
                else:
                    if token not in extra_feedJson:
                        extra_feedJson[token] = []
                        last_resample_time[token] = timest

                    extra_feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
                    if len(extra_feedJson[token]) > 10:
                        extra_feedJson[token].pop(0)

    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    
    while True:
        if not feedJson:
            await asyncio.sleep(1)
            continue

        with resampling_lock:
            for token, ticks in feedJson.items():
                try:
                    if ticks:
                        # Create a DataFrame with the new ticks
                        df_new = pd.DataFrame(ticks)
                        df_new["tt"] = pd.to_datetime(df_new["tt"])
                        df_new.set_index("tt", inplace=True)

                        # Determine the current resample interval
                        current_resample_interval = df_new.index.max().floor(resample_frequency)

                        # Initialize or update the resampled DataFrame
                        if token not in resampled_data:
                            # Initialize the DataFrame with the first resample
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()
                            resampled_data[token] = df_resampled
                            last_resample_time[token] = df_resampled.index.max()
                        else:
                            # Resample the new ticks
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()

                            # Update the existing resampled DataFrame with new data
                            for idx, row in df_resampled.iterrows():
                                if idx in resampled_data[token].index:
                                    resampled_data[token].loc[idx, 'high'] = max(resampled_data[token].loc[idx, 'high'], row['high'])
                                    resampled_data[token].loc[idx, 'low'] = min(resampled_data[token].loc[idx, 'low'], row['low'])
                                    resampled_data[token].loc[idx, 'close'] = row['close']
                                else:
                                    resampled_data[token].loc[idx] = row

                            # Update the last resampled time
                            last_resample_time[token] = df_resampled.index.max()

                        long_stop, short_stop, direction = chandelier_exit_numba(resampled_data[token]['open'].values,resampled_data[token]['high'].values,resampled_data[token]['low'].values,resampled_data[token]['close'].values)

                        resampled_data[token]['long_stop'] = long_stop
                        resampled_data[token]['short_stop'] = short_stop
                        resampled_data[token]['ce_direction'] = direction

                    ##################################################################################
                    # if len(feedJson[token]) > 50:
                    #     feedJson[token].pop(0)    
                    ##################################################################################
                    feedJson[token] = []

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")

        await asyncio.sleep(0.01)  
        
def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming `api` is defined and starts the WebSocket connection
async def connect_and_subscribe():
    global feed_opened
    api.start_websocket(
        order_update_callback=event_handler_order_update,
        subscribe_callback=event_handler_feed_update,
        socket_open_callback=open_callback
    )
    while not feed_opened:
        await asyncio.sleep(1)  # Wait for initial connection
    api.subscribe(['NSE|26009'])

async def main():
   
    resample_task = asyncio.create_task(resample_ticks())
    trading_task = asyncio.create_task(trading_logic())
    connect_task = asyncio.create_task(connect_and_subscribe())  
    update_symbols_task = asyncio.create_task(update_atm_symbols())
    
    await asyncio.gather(connect_task, resample_task, trading_task, update_symbols_task)

def set_trading_active(value):
    global trading_active
    trading_active = value
    print(f"Trading active set to {trading_active}")

# Get the current event loop
loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()
    asyncio.create_task(main())
else:
    loop.run_until_complete(main())

Error in update_atm_symbols: '26009'
last 5 bar:  20.320000000001166
current length:  15.55000000000291
current time:  2024-08-02 11:24:00.352059
last 5 bar:  19.040000000000873
current length:  17.75
current time:  2024-08-02 11:25:00.027123
last 5 bar:  21.520000000001165
current length:  28.19999999999709
current time:  2024-08-02 11:26:00.024895
Short Entry Signal for BANKNIFTY07AUG24P51500 at 2024-08-02 11:26:00.025194
last 5 bar:  20.250000000001457
current length:  24.05000000000291
current time:  2024-08-02 11:27:00.084504
last 5 bar:  22.219999999999708
current length:  25.549999999995634
current time:  2024-08-02 11:28:00.069256
last 5 bar:  21.919999999998254
current length:  14.049999999995634
current time:  2024-08-02 11:29:00.005800
last 5 bar:  20.779999999998836
current length:  12.05000000000291
current time:  2024-08-02 11:30:00.086097
last 5 bar:  18.41999999999971
current length:  16.400000000001455
current time:  2024-08-02 11:31:00.018130
last 5 bar:  18.189999999

In [ ]:
op_chain = api.get_option_chain(exchange='NFO', tradingsymbol=state.ce_trading_symbol, strikeprice=atm_strike, count =10)
tokens = [item['token'] for item in op_chain['values']]
subscriptions = [f"NFO|{token}" for token in tokens]
# Subscribe to each token
for sub in subscriptions:
    api.subscribe(sub)

# Print the subscriptions for verification
print("Subscribed to:", subscriptions)


realtime_exit_checker(current_positions, entry_instruments, profit_target, stop_loss, extra_feedJson)

In [ ]:
def append_to_csv(trade_log):
    with open('trading_log.csv', 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow()

import pandas as pd

# Assuming resampled_data has the format {'token_name': DataFrame}
token, df = next(iter(resampled_data.items()))  # Get the token name and DataFrame

df.to_csv(f"{token}.csv", index=True)   # Save the DataFrame to a CSV file (include the index)
print(f"Saved data to {token}.csv")   

In [11]:
resampled_data

{'26009':                          open      high       low     close  long_stop  \
 tt                                                                       
 2024-08-02 11:18:00  51491.05  51494.00  51482.40  51483.65        NaN   
 2024-08-02 11:19:00  51479.60  51484.30  51460.15  51465.85        NaN   
 2024-08-02 11:20:00  51465.40  51466.85  51451.05  51456.00  51453.775   
 
                      short_stop  ce_direction  
 tt                                             
 2024-08-02 11:18:00         NaN           0.0  
 2024-08-02 11:19:00         NaN           0.0  
 2024-08-02 11:20:00   51468.075           0.0  }

Retracement condition met, entering trade.
Short Entry Signal for BANKNIFTY07AUG24P51500 at 2024-08-02 11:53:23.447697
Retracement condition met, entering trade.
Long Entry Signal for BANKNIFTY07AUG24C51400 at 2024-08-02 12:54:02.547937
